# Look-ahead · Reward  `[EVAL]`

**RQ-i on the eval rubrics: does K-turn look-ahead change what the policy scores?** The look-ahead lever isolated — the *same* optimizer at K=0 vs K=5 (`PTO_LA0` vs `PTO_LA5`, `GRPO_LA0` vs `GRPO_LA5`), persona-paired at every iteration both arms reached, on all eight instruments (+ Q1 / Q2 separately), under **both graders side by side**: the training oracle (`gpt-4o-mini`, which WAS the reward) and the held-out judge (`claude-haiku-4-5`). This family owns the cross-K *reward* story; the behaviour channels live in `lookahead/behaviour`, the cross-grader transfer/retention in `lookahead/transfer`, the mechanism in `lookahead/mechanism`, and the cost axis in `compute/cost`.

**Ported 2026-08-18** from `7_Stats` §4c (`k_means_by_iter`, `k_paired_by_method`, `k_trajectory_Q1Q2` — previously rendered per grader under `results/L5/…/7_stats/<judge>/`; now one table/figure holding both graders) and from the look-ahead paper's promoted generators (`eda_analysis.lookahead` ← `k_contrast_headline.py` rubric part + `cross_k_multijudge.py` §3–§4). Statistics unchanged; the artifact names drop the script prefix (`k_contrast_headline_table1` → `k_table1`).

**Conventions (every caption restates them).**
- **Sign: `+ ⇒ K=0 higher`** (K=0 minus K=5). A positive Δ on a higher-is-better rubric means look-ahead *cost* score; on `MICI` (lower-is-better) `+` means K=0 is *worse*.
- **Pairing unit: `persona_id`** — the 96 patient personas recur in every model state (the trainer reshuffles them each iteration, so `file_index` is not a pairing key).
- **Iteration 0 = two independent base draws** of the same 96 personas (the K=0 arm's base vs the K=5 arm's base): the noise floor, never dropped.
- **Censoring: `GRPO_LA5` ends at iteration 5** (its full budget); PTO K=5 runs to 10. Matched iterations are read off the data. ⚠ Matched *iteration* is not matched *budget* — a K=5 iteration costs ~1.9× a K=0 one; `compute/cost` re-asks the question at matched GPU-hours.
- **The two graders' raw scores are never averaged** — every table carries a `judge` column (or `primary_*` / `judge_*` pairs).
- Bootstrap CIs are seeded with `constants.BOOT_SEED`; means / *dz* / *p* / *n* are exact w.r.t. the paper fixture, CI bounds agree to ~0.02.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)

import eda_analysis
from eda_analysis import exports, plotting, stats, lookahead
from eda_analysis.constants import BOOT_SEED, DISPLAY_NAMES

cfg = eda_analysis.EdaConfig(family="lookahead/reward", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # reset_results wiped the banner notebook_setup wrote — re-stamp it

## 0 · Both graders' score frames  `[EVAL]`
**Purpose.** This family is judge-invariant: it loads every grader in the score lake through `scores_by_judge` (primary first) and keeps them apart. `PRIMARY` / `HELDOUT` are the short labels the tables carry in their `judge` column; the arm list is the four-arm default (no K filter — the retired `L0`/`L5` VIEW used to hide one K arm from the other).

In [ ]:
SC = eda_analysis.scores_by_judge(S)          # {'gpt-4o-mini': scores_long, 'claude-haiku-4-5': scores_long}
JUDGES = list(SC)
PRIMARY, HELDOUT = JUDGES[0], (JUDGES[1] if len(JUDGES) > 1 else None)
K_ARMS = [a for a in ["PTO_LA0", "PTO_LA5", "GRPO_LA0", "GRPO_LA5"] if a in set(SC[PRIMARY].arm.unique())]
PAL = plotting.arm_palette(sorted(SC[PRIMARY].arm.unique()))
for j, sc in SC.items():
    print(f"{j:>18}: {sc.shape} | arms {sorted(sc.arm.unique())} | iters "
          + ", ".join(f"{a}:0..{int(sc[sc.arm == a].iteration.max())}" for a in K_ARMS))
assert HELDOUT is not None, "lookahead/reward needs the primary AND a held-out judge on disk"
JUDGE_TITLE = {PRIMARY: f"{PRIMARY} (training oracle)", HELDOUT: f"{HELDOUT} (held-out judge)"}

## 1 · Levels, paired test and the reward curve — the `7_Stats` §4c artifacts, both graders  `[EVAL]`
**Purpose.** The three tracked RQ-i artifacts, unchanged in statistic and name, now with a `judge` column instead of a `<judge>/` folder: `k_means_by_iter` (where the two K arms sit, per method × rubric × iteration — an UNPAIRED difference of arm means), `k_paired_by_method` (the persona-paired Wilcoxon + *dz* + Holm — **Holm across the rubrics within each (judge, method, iteration)**), and `k_trajectory_Q1Q2` (both K arms of both methods on the primary metric, one panel per grader).

**Read it against `compute/cost`.** Both K=5 arms are trained well past a prefix (PTO to iter 10, GRPO to iter 5), so the matched-iteration comparison is real — but a K=5 iteration costs ~1.9× a K=0 one, and the two GRPO arms are budget-matched to within ~3% *despite* the iteration gap. Quote both axes, or say which one you mean.

In [ ]:
# ── LEVELS: K=0 vs K=5 arm means per (judge, method, rubric, iteration) ─────────────────────
KM = pd.concat([M.assign(judge=j) for j in JUDGES
                for M in (stats.k_means_by_iter(SC[j], m) for m in ["PTO", "GRPO"]) if not M.empty],
               ignore_index=True)
KM = KM[["judge"] + [c for c in KM.columns if c != "judge"]].round(4)
print("=== K=0 vs K=5 arm MEANS per iteration (levels; + delta => K0 higher), both graders ===")
display(KM[KM.metric == "Q1Q2"])
exports.save_table(KM, "k_means_by_iter", caption=(
    "RQ-i LEVELS, both graders (column `judge`: gpt-4o-mini = training oracle, claude-haiku-4-5 = held-out): "
    "K=0 vs K=5 arm means per method x rubric x iteration, with the unpaired delta (+ => K=0 higher) and each "
    "side's n (96 conversations per cell). Iterations the K=5 arm never reached keep mean_K5/delta NaN (n_K5=0) — "
    "GRPO_LA5 is censored at iteration 5, PTO_LA5 reaches 10; K=0 continuing past the K=5 arm's last iteration is "
    "part of the look-ahead answer. Read dz/p off k_paired_by_method, NOT here: delta is a difference of arm means "
    "(equal to the paired mean_delta when both cells are complete, but carrying none of the persona pairing's "
    "precision). Same statistic as the retired results/L5/tables/7_stats/<judge>/k_means_by_iter (one table per grader)."))

# ── TEST: K0 - K5 within each method, persona-paired, Holm across rubrics within (judge, method, iteration) ──
frames = []
for j in JUDGES:
    for method in ["PTO", "GRPO"]:
        CMP = stats.paired_k_comparison(SC[j], method)
        if not CMP.empty:
            frames.append(CMP[CMP.iteration > 0].assign(judge=j))
KP = pd.concat(frames, ignore_index=True)
KP = KP[["judge", "method", "iteration", "metric", "n", "mean_delta", "dz", "p", "p_holm"]].round(4)
print("=== K0 - K5 within each method, persona-paired (+ => K0 higher), both graders ===")
display(KP[KP.metric == "Q1Q2"])
exports.save_table(KP, "k_paired_by_method", caption=(
    "RQ-i TEST, both graders (column `judge`): K0 - K5 within each method at every matched iteration >= 1, "
    "persona-paired (persona_id, n = 96) Wilcoxon + Cohen's dz + Holm. + => K=0 higher, so a positive delta means "
    "look-ahead COST score (on MICI, lower-is-better, + means K=0 is WORSE). Holm scope: p_holm is corrected across "
    "the rubrics WITHIN each (judge, method, iteration) contrast, not across iterations — the long tables in section 2 "
    "use the other family (Holm across iterations), so p agrees cell-for-cell with them while p_holm need not. "
    "GRPO_LA5 is censored at iteration 5 (GRPO rows stop there); PTO to 10. Matched-ITERATION, not matched-budget: "
    "a K=5 iteration costs ~1.9x a K=0 one (compute/cost), and the two GRPO arms are budget-matched within ~3%. "
    "Same statistic as the retired results/L5/tables/7_stats/<judge>/k_paired_by_method."))

# ── FIGURE: the reward curve, both K arms of both methods, one panel per grader ─────────────
fig, axes = plt.subplots(1, len(JUDGES), figsize=(7.4 * len(JUDGES), 4.6), sharey=False, squeeze=False)
for ax, j in zip(axes[0], JUDGES):
    d = SC[j][(SC[j].questionnaire == "Q1Q2") & (SC[j].arm.isin(K_ARMS))]
    sns.lineplot(d, x="iteration", y="score", hue="arm", palette=PAL, marker="o",
                 errorbar=("ci", 95), seed=BOOT_SEED, ax=ax)
    base = d[d.is_base]
    if not base.empty:
        b0 = float(base.score.mean())
        ax.axhspan(b0 - S.ORACLE_NOISE, b0 + S.ORACLE_NOISE, color="grey", alpha=0.15)
        ax.text(0.02, b0 + S.ORACLE_NOISE, " ~oracle-noise band around base", fontsize=7, va="bottom", color="grey")
    ax.set_title(f"{DISPLAY_NAMES.get('Q1Q2', 'Q1Q2')} — grader: {JUDGE_TITLE[j]}", fontsize=10)
    ax.set_xlabel("iteration (0 = base)"); ax.set_ylabel(f"{DISPLAY_NAMES.get('Q1Q2', 'Q1Q2')} (mean, 95% CI over 96 personas)")
    ax.set_xticks(range(0, int(d.iteration.max()) + 1))
    plotting.relabel_legend(ax)
fig.suptitle("RQ-i in one frame: both look-ahead arms of both methods (K=5 lines stop at their last scored iteration)",
             fontsize=10, y=1.02)
fig.tight_layout()
exports.save_fig(fig, "k_trajectory_Q1Q2", caption=(
    "RQ-i in one frame, one panel per grader (left = gpt-4o-mini training oracle, right = claude-haiku-4-5 held-out "
    "judge; separate y-scales — the held-out judge sits 1.2-1.7 points lower and the two are never averaged): both "
    "look-ahead arms of both methods overlaid on the primary metric Q1+Q2 (mean +/- 95% CI over the 96 personas; grey "
    "band = +/-0.10 oracle-repeatability around the pooled base). The K=5 lines STOP at their last scored iteration "
    "(PTO 10, GRPO 5 — GRPO_LA5 is censored there) — everything to the right is K=0 only, so compare the arms in the "
    "overlapping region and read the tail as 'K=0 kept training', not as a K difference. For GRPO that tail is NOT "
    "free: on the compute axis (compute/cost) GRPO's two arms cost the same total GPU-hours."))
plt.show()

## 2 · The paired K contrast, long form — per (grader, method), all rubrics, iterations 0..N  `[EVAL]`
**Purpose.** The paper's headline machinery (`lookahead.paired_k_frames`): one row per (judge, method, rubric, iteration) with the two arm levels (mean ± SE), the persona-paired `mean_delta`, Cohen's *dz*, a persona-bootstrap 95 % CI, Wilcoxon *p*, and `p_holm` — here **Holm across iterations 0..N within (judge, method, rubric)** (the paper's family; a different family from §1's `k_paired_by_method`, so `p` agrees cell-for-cell while `p_holm` need not). Iteration 0 is kept as the base-vs-base noise floor.

Saved as one long table per (method, grader) — `k_paired_<method>_<judge>` — plus the concatenation `k_paired_long`; `k_table1` (+ `_Q1`, `_Q2`, `_MICI`, `_PCT`) is the compact `Δ (dz)★` cell layout the write-up quotes; `k_summary` counts, per (grader, method, rubric), how many iterations clear Holm in each direction.

In [ ]:
FR = lookahead.paired_k_frames(SC, holm_family="iterations")     # judge, method, metric, iteration(0..N), n, levels, delta, dz, CI, p, p_holm, sig
print("paired K frames:", FR.shape, "| judges:", list(dict.fromkeys(FR.judge)), "| metrics:", list(dict.fromkeys(FR.metric)))
display(FR[FR.metric == "Q1Q2"].round(3))

for method in lookahead.METHODS:
    for j in JUDGES:
        d = FR[(FR.method == method) & (FR.judge == j)].drop(columns=["judge", "method"])
        exports.save_table(d, f"k_paired_{method.lower()}_{j}", caption=(
            f"**{method}, K=0 vs K=5, grader = {JUDGE_TITLE[j]}.** Persona-paired contrast per rubric x iteration: "
            f"levels (mean +/- SE over personas per arm), mean_delta, Cohen's dz, persona-bootstrap 95% CI, Wilcoxon p, "
            f"p_holm. {lookahead.SIGN_NOTE} {lookahead.HOLM_NOTE} MICI is lower-better (+ = K=0 WORSE there). "
            + (lookahead.CENSOR_NOTE if method == "GRPO" else "PTO_LA5 reaches iteration 10 (matched to 10).")
            + f" Bootstrap CIs seeded with BOOT_SEED={BOOT_SEED} (means/dz/p exact w.r.t. the paper fixture "
            f"k_contrast_headline_{method.lower()}_{'primary' if j == PRIMARY else 'heldout'}; CI bounds agree to ~0.02)."))
exports.save_table(FR, "k_paired_long", caption=(
    f"**All four method x grader K contrasts, long form** (columns `judge`, `method`). {lookahead.SIGN_NOTE} "
    f"{lookahead.HOLM_NOTE} {lookahead.CENSOR_NOTE} The union of the four k_paired_<method>_<judge> tables "
    f"(paper fixture: k_contrast_headline_all_long)."))

# ── Table 1: the compact Δ (dz)★ layout on Q1+Q2, then Q1 / Q2 / MICI / PCT ────────────────
T1 = lookahead.k_table1(FR, "Q1Q2")
print("=== Table 1: paired K=0 - K=5 on Q1+Q2, cell = mean_delta (dz) + Holm stars ==="); display(T1)
exports.save_table(T1, "k_table1", caption=(
    f"**Table 1 — paired K=0 - K=5 on the training reward Q1+Q2, by iteration, under both graders** (columns "
    f"'<method> · <judge>'; gpt-4o-mini = training oracle, claude-haiku-4-5 = held-out). Cell = mean_delta (Cohen's dz) "
    f"with Holm stars (* <.05, ** <.01, *** <.001; {lookahead.HOLM_NOTE.rstrip('.')}). {lookahead.SIGN_NOTE} "
    f"{lookahead.CENSOR_NOTE} '—' = no matched K=5 model state (GRPO after 5). Iteration 0 = two independent base "
    f"draws (noise floor)."))
for m in ["Q1", "Q2", "MICI", "PCT"]:
    exports.save_table(lookahead.k_table1(FR, m), f"k_table1_{m}", caption=(
        f"**Paired K=0 - K=5 on {DISPLAY_NAMES.get(m, m)}, by iteration, both graders** (columns '<method> · <judge>'). "
        f"Cell = mean_delta (dz) + Holm stars. {lookahead.SIGN_NOTE} {lookahead.HOLM_NOTE} {lookahead.CENSOR_NOTE}"
        + (" MICI is LOWER-better: + means K=0 is MORE MI-inconsistent (worse)." if m == "MICI" else "")
        + (" PCT is a 0-1 rate (deltas ~3x smaller than the point-scale rubrics; no +/-0.10 band)." if m == "PCT" else "")))

# ── Summary per (grader, method, rubric) across iterations ────────────────────────────────
SUM = lookahead.k_summary(FR)
print("=== Summary: iterations clearing Holm per (judge, method, rubric) ==="); display(SUM[SUM.metric.isin(["Q1Q2", "MICI"])])
exports.save_table(SUM, "k_summary", caption=(
    f"**Per (grader, method, rubric) summary of the K contrast across iterations** (column `judge`). n_sig_K0_higher / "
    f"n_sig_K5_higher count iterations with Holm p<.05 and delta >0 / <0; the *_better columns flip the sign for "
    f"lower-better rubrics (MICI). mean_delta_iters1toN / mean_dz_iters1toN average the per-iteration paired deltas "
    f"over TRAINED iterations only; base_delta / base_dz are the iteration-0 base-vs-base draw; max_abs_dz is the "
    f"largest |dz| over all iterations incl. 0. {lookahead.SIGN_NOTE} {lookahead.HOLM_NOTE} {lookahead.CENSOR_NOTE}"))

## 3 · Where the four arms sit — levels under both graders, and the headline figures  `[EVAL]`
**Purpose.** The un-paired LEVELS the contrast is computed on (`k_levels`: Q1+Q2 arm means by iteration, one column per `<arm> · <judge>`; `k_levels_long`: every arm × rubric × iteration with mean / sd / SE), and the two figure families the write-up leads with: `k_headline_q1q2` (four-arm level curves per grader over the paired K=0 − K=5 delta strip, ±0.10 band, filled = Holm p<.05) and `k_delta_grid_<judge>` (the delta strip on every rubric, one 3×3 grid per grader).

⚠ Read *dz* / *p* off the contrast tables, never off the levels — the levels carry no pairing.

In [ ]:
LV = lookahead.k_levels(SC)              # {'levels': wide Q1Q2 (iteration x '<arm> · <judge>'), 'levels_long': long}
LEVELS, LEVELS_LONG = LV["levels"], LV["levels_long"]
print("=== Q1+Q2 arm means by iteration, both graders ==="); display(LEVELS.round(3))
exports.save_table(LEVELS, "k_levels", caption=(
    "**Q1+Q2 arm means by iteration under both graders** (columns '<arm> · <judge>'; 96 conversations per cell; "
    "iteration 0 = each arm's own base draw). NOT paired — read dz/p off the contrast tables (k_paired_*). "
    f"{lookahead.CENSOR_NOTE} GRPO_LA0 continues to iteration 10 unmatched; PTO_LA5 reaches 10. NaN = the arm has no "
    "such iteration. Never average the two graders' columns (the held-out judge sits 1.2-1.7 points lower)."))
exports.save_table(LEVELS_LONG, "k_levels_long", caption=(
    "**Arm x rubric x iteration levels (n, mean, sd ddof=1, SE over the 96 personas) under both graders** "
    "(column `judge`). The long form behind k_levels and the level rows of the k_headline_q1q2 figure. "
    f"{lookahead.CENSOR_NOTE}"))

fig = plotting.k_headline_fourarm(LEVELS_LONG, FR, metric="Q1Q2", palette=PAL, oracle_noise=S.ORACLE_NOISE)
exports.save_fig(fig, "k_headline_q1q2", caption=(
    "THE headline, one column per grader (left = gpt-4o-mini training oracle, right = claude-haiku-4-5 held-out judge; "
    "row y-limits shared across the two so they read on one scale). TOP: the four arms' Q1+Q2 level by iteration (mean "
    "+/- SE over 96 personas; K=0 solid + circle, K=5 dashed + square, each arm's own base dotted; 'GRPO K=5 ends' marks "
    "the censoring at iteration 5). BOTTOM: the persona-paired K=0 - K=5 delta per method (PTO left-shifted, GRPO "
    "right-shifted; 95% CI whiskers; filled = Holm p<.05 across iterations, hollow = n.s.) over the +/-0.10 "
    "oracle-repeatability band. Sign: + => K=0 higher (look-ahead cost score). Iteration 0 = two independent base "
    "draws (noise floor). Paper fixture: k_contrast_headline_fig_q1q2."))
plt.show()

for j in JUDGES:
    fig = plotting.k_delta_grid(FR, j, palette=PAL, oracle_noise=S.ORACLE_NOISE)
    exports.save_fig(fig, f"k_delta_grid_{j}", caption=(
        f"Paired K=0 - K=5 by iteration on EVERY rubric (3x3 grid), grader = {JUDGE_TITLE[j]}. Point-scale rubrics get "
        f"the +/-0.10 oracle-repeatability band; PCT / MICI (0-1 rates) get a bare zero line and say so; lower-better "
        f"rubrics are flagged (MICI: + = K=0 more MI-inconsistent = worse). PTO left-shifted, GRPO right-shifted; 95% CI "
        f"whiskers; filled = Holm p<.05 across iterations 0..N within (judge, method, rubric). Sign: + => K=0 higher. "
        f"Paired on persona_id (n = 96). {lookahead.CENSOR_NOTE} PTO matched to 10. Paper fixture: "
        f"k_contrast_headline_fig_grid_{'primary' if j == PRIMARY else 'heldout'}."))
    plt.show()

## 4 · K × method — the difference-in-differences and the method gap at each K  `[EVAL]`
**Purpose.** Does look-ahead help one optimizer more than the other? `k_did`: the persona-level DiD `(PTO_LA0 − GRPO_LA0) − (PTO_LA5 − GRPO_LA5)` at every iteration all four arms share (0..5 — GRPO_LA5 stops at 5, so 6..10 is **not** estimable; iteration 0 = four independent base draws). **Sign: `+ ⇒ PTO's lead over GRPO is LARGER at K=0 than at K=5** (equivalently, look-ahead helps GRPO more than PTO); on MICI the sign reads the other way round. `k_method_gap`: its ingredients — `PTO_LA{K} − GRPO_LA{K}` at each K and every matched iteration (**+ ⇒ PTO higher**; on MICI + favours GRPO — read `favours`).

Two figures: `k_did` (the method gap at each K over the DiD, one column per grader) and `k_contrast_both_judges` (the K contrast on Q1+Q2 and MICI under BOTH graders on one axis — the primary solid + CI ribbon, the held-out dotted + CI bars, stars = Holm p<.05 under either grader).

In [ ]:
DID = lookahead.did_by_iter(SC)
GAP = lookahead.method_gap_by_iter(SC)
print("=== K x method DiD (+ => PTO's lead larger at K=0), Q1+Q2, both graders ==="); display(DID[DID.metric == "Q1Q2"].round(3))
exports.save_table(DID, "k_did", caption=(
    "**K x method interaction (difference-in-differences) per persona**, iterations 0..5 (the only iterations all "
    "four arms share — GRPO_LA5 is censored at 5, so the interaction is NOT estimable at 6..10; iteration 0 = four "
    "independent base draws, a noise-floor row). gap_K0 = mean(PTO_LA0 - GRPO_LA0), gap_K5 = mean(PTO_LA5 - GRPO_LA5) "
    "(+ => PTO higher); did = gap_K0 - gap_K5 computed persona by persona (persona_id, n = 96), so + => PTO's lead over "
    "GRPO is LARGER at K=0 than at K=5 (equivalently, look-ahead helps GRPO more than PTO). On MICI the sign reads the "
    "other way round (lower is better). Column `judge` names the grader (gpt-4o-mini = training oracle; "
    "claude-haiku-4-5 = held-out). dz = mean/SD of the per-persona DiD; CI = persona bootstrap; p = Wilcoxon; p_holm = "
    "Holm across iterations 0..5 within (grader, metric); p_holm_rubrics = Holm across the 9 rubrics within (grader, "
    "iteration). Cross-check: held-out Q1Q2 iteration 5 dz ~ 0.525. Paper fixture: cross_k_multijudge_did."))
exports.save_table(GAP, "k_method_gap", caption=(
    "**The method gap at each look-ahead K under both graders.** PTO_LA{K}_In - GRPO_LA{K}_In at every matched "
    "iteration (iteration 0 = two independent base draws). Sign: + => PTO higher; on MICI (lower-is-better) + favours "
    "GRPO — read `favours`. Paired on persona_id (n = 96). Column `judge` names the grader (gpt-4o-mini = training "
    "oracle; claude-haiku-4-5 = held-out). CI = persona bootstrap; p = Wilcoxon; p_holm = Holm across ITERATIONS within "
    "(grader, K, metric); p_holm_rubrics = Holm across the 9 rubrics within (grader, K, iteration) (the tracked "
    "method_paired_by_K convention, method/contrast). GRPO_LA5 is censored at iteration 5, so the K=5 rows stop there "
    "while the K=0 rows run to 10. Paper fixture: cross_k_multijudge_method_gap."))

fig = plotting.k_did(DID, GAP, metric="Q1Q2", judge_titles={PRIMARY: f"{PRIMARY} (oracle)", HELDOUT: f"{HELDOUT} (held-out)"})
exports.save_fig(fig, "k_did", caption=(
    "The method gap PTO - GRPO at each K (top: K=0 black solid + circle, K=5 green dashed + square; persona-bootstrap "
    "95% CI ribbons; stars = Holm p<.05 across iterations) over the K x method DiD (bottom: gap(K=0) - gap(K=5) per "
    "persona, dz annotated per iteration), one column per grader (left = gpt-4o-mini training oracle, right = "
    "claude-haiku-4-5 held-out; row y-limits shared). Signs: gap + => PTO higher; DiD + => PTO's lead over GRPO is "
    "larger at K=0 than at K=5. The DiD is estimable only while all four arms run (iterations 0..5 — GRPO_LA5 is "
    "censored at 5); the K=0 gap continues to 10. n = 96 personas. Paper fixture: cross_k_multijudge_fig_did."))
plt.show()

fig = plotting.k_contrast_both_judges(FR, metrics=("Q1Q2", "MICI"), palette=PAL, primary=PRIMARY, heldout=HELDOUT,
                                      primary_label=f"training oracle ({PRIMARY})",
                                      heldout_label=f"held-out judge ({HELDOUT})")
exports.save_fig(fig, "k_contrast_both_judges", caption=(
    "The K contrast under BOTH graders on one axis — rows Q1+Q2 / MICI, columns PTO / GRPO. Primary (gpt-4o-mini "
    "training oracle): solid line + filled circle + CI ribbon; held-out (claude-haiku-4-5): dotted line + open circle + "
    "CI bars; star = Holm p<.05 (across iterations 0..N) under either grader. Sign: + => K=0 higher; MICI is "
    "lower-is-better, so on MICI + favours K=5. Paired on persona_id (n = 96); iteration 0 = two independent base "
    "draws. GRPO_LA5 is censored at iteration 5 (the GRPO panels stop there); PTO to 10. Paper fixture: "
    "cross_k_multijudge_fig_kcontrast."))
plt.show()

## 5 · Endpoint contrasts under both graders  `[EVAL]`
**Purpose.** The handful of two-model contrasts the write-up quotes, each under the training oracle (`primary_*`) AND the held-out judge (`judge_*`), on every rubric: the K=0 headline (`PTO_LA0` vs `GRPO_LA0` at their endpoints), the K=5 endpoints, the K lever at each method's endpoint / matched iteration, and GRPO's K=5 endpoint against GRPO_LA0's endpoint and against its BEST iteration by mean Q1+Q2 under each grader (one extra pair when the two graders disagree on that best). Endpoints are read off the data (each arm's last iteration), so the pairs follow the arms. `*_p_holm` = Holm across the rubrics within a pair (the tracked `compare_two_models` convention). Sign: `+ ⇒ A (left model) higher`; on MICI + favours B — read `favours_*`.

In [ ]:
END = lookahead.endpoint_contrasts(SC, primary=PRIMARY, heldout=HELDOUT)
print("=== endpoint contrasts, both graders (Q1+Q2 rows) ==="); display(END[END.metric == "Q1Q2"].round(3))
exports.save_table(END, "k_endpoints", caption=(
    "**Endpoint contrasts under both graders** (A - B as named in `pair`; + => A higher; on MICI, lower is better, so + "
    "favours B — read `favours_*`, where A/B are the pair's left/right model). primary_* = training oracle gpt-4o-mini; "
    "judge_* = held-out claude-haiku-4-5; same_sign / judge_ci_excl0 = whether the held-out judge agrees in sign / has a "
    "CI excluding 0. Paired on persona_id (n = 96); CI = persona bootstrap; p = Wilcoxon; *_p_holm = Holm across the "
    "rubrics within a pair (the tracked compare_two_models convention). Endpoints are each arm's LAST iteration on disk "
    "(GRPO_LA5 is censored at 5, the other three reach 10); GRPO_LA0's best iteration is chosen by mean Q1+Q2 under "
    "each grader (an extra pair appears when the graders disagree). Paper fixture: cross_k_multijudge_endpoints "
    "(judge_ci_excl0 can flip on a CI bound within bootstrap noise — one flag differs at BOOT_SEED)."))

## 6 · The number ledger  `[EVAL]`
Every number the write-up may quote from this family, as `{dotted.key: {value, source, note}}` → `tables/k_numbers.json` (`k.<method>.<metric>.iter<n>.<judge>`, `base_vs_base.*`, `summary.*`, `level.Q1Q2.*`, `did.*`, `method_gap.K<K>.*`, `endpoint.<pair>.<metric>`, `conventions`). Papers cite `results/lookahead/reward/tables/k_numbers.json :: <key>`; the `source` of each entry names the producing frame + row.

In [ ]:
NUM = lookahead.lookahead_numbers(FR, summary=SUM, levels_long=LEVELS_LONG, did=DID, method_gap=GAP, endpoints=END,
                                  oracle_noise=S.ORACLE_NOISE)
# the paper's cross_k_multijudge.json also records which GRPO_LA0 iteration the "best" endpoint pairs used
NUM["grpo_la0_best_iter_by_q1q2"] = {
    "value": {"primary": lookahead.best_iteration(SC[PRIMARY], "GRPO_LA0", "Q1Q2"),
              "heldout": lookahead.best_iteration(SC[HELDOUT], "GRPO_LA0", "Q1Q2")},
    "source": "lookahead.best_iteration(scores_by_judge[<judge>], 'GRPO_LA0', 'Q1Q2') — the iteration behind the "
              "'GRPO_LA0 best' pairs of k_endpoints",
    "note": f"primary = {PRIMARY} (training oracle); heldout = {HELDOUT} (held-out judge); best = highest mean Q1+Q2 over the 96 personas"}
print(f"ledger: {len(NUM)} keys")
exports.save_numbers("k_numbers", NUM, caption=(
    "Number ledger for lookahead/reward: every quotable cell of the paired K contrast (k.*, base_vs_base.*, summary.*), "
    "the Q1+Q2 levels (level.*), the K x method DiD (did.*), the method gap at each K (method_gap.*) and the endpoint "
    "contrasts (endpoint.*), each with its producing frame + row as `source`; `conventions` restates sign (+ => K=0 "
    "higher), pairing (persona_id, n = 96), Holm family, the +/-0.10 band, iteration 0 = two independent base draws, "
    "and the GRPO_LA5 censoring at iteration 5. Reproduces the paper's k_contrast_headline.json (rubric keys) + the "
    "did/method_gap/endpoint keys of cross_k_multijudge.json (CI bounds to bootstrap noise)."))

In [ ]:
exports.prune_orphan_captions(); exports.build_index()